In [1]:
# CELL 1 — Phase 4 config and dataset loading
import numpy as np
import torch
import torch.nn as nn
import math
import time
import json
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

N_QUBITS = 4
ENTANGLING_LAYERS = 2
D_MODEL = 64
D_FF = 128
N_HEADS = 4
N_TOKENS = 225
PATCH_SIDE = 15
TRAIN_BATCH_SIZE = 32
EPOCHS = 50
LR = 2e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SEEDS = [42, 43, 44]
NOISE_STD = 0.02  # midpoint of locked 0.01-0.03 range

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

EXTENSION_DATASETS = ["Salinas", "KSC", "Botswana"]
datasets = {}
for name in EXTENSION_DATASETS:
    d = np.load(f"preprocessed/{name}.npz")
    datasets[name] = {
        "train_tokens": torch.tensor(d["train_tokens"], dtype=torch.float32),
        "train_labels": torch.tensor(d["train_labels"] - 1, dtype=torch.long),
        "val_tokens": torch.tensor(d["val_tokens"], dtype=torch.float32),
        "val_labels": torch.tensor(d["val_labels"] - 1, dtype=torch.long),
        "test_tokens": torch.tensor(d["test_tokens"], dtype=torch.float32),
        "test_labels": torch.tensor(d["test_labels"] - 1, dtype=torch.long),
    }
    k_dim = datasets[name]["train_tokens"].shape[-1]
    n_classes = int(datasets[name]["train_labels"].max().item()) + 1
    datasets[name]["k_dim"] = k_dim
    datasets[name]["n_classes"] = n_classes
    print(f"{name}: k={k_dim}, classes={n_classes}, "
          f"train={datasets[name]['train_tokens'].shape}")

Using device: cuda
Salinas: k=10, classes=16, train=torch.Size([5412, 225, 10])
KSC: k=10, classes=13, train=torch.Size([521, 225, 10])
Botswana: k=10, classes=14, train=torch.Size([324, 225, 10])


In [2]:
# CELL 2 — Geometric augmentation (post-PCA, mathematically equivalent to pre-PCA)
# + noise augmentation (DEVIATION: applied in PCA-component space, not raw band
# space, because raw normalized patches were not retained from Phase 1)

def augment_batch(tokens_flat, k_dim, rng, noise_std=NOISE_STD):
    """
    tokens_flat: (N, 225, k) -> reshape to (N, 15, 15, k) grid, augment, reflatten.
    """
    n = tokens_flat.shape[0]
    grid = tokens_flat.reshape(n, PATCH_SIDE, PATCH_SIDE, k_dim).clone()

    for i in range(n):
        if rng.random() < 0.5:
            grid[i] = torch.flip(grid[i], dims=[0])
        if rng.random() < 0.5:
            grid[i] = torch.flip(grid[i], dims=[1])
        k_rot = rng.integers(0, 4)
        if k_rot > 0:
            grid[i] = torch.rot90(grid[i], k=k_rot, dims=[0, 1])

    grid = grid + torch.randn_like(grid) * noise_std  # DEVIATION: noise in PCA space
    return grid.reshape(n, PATCH_SIDE * PATCH_SIDE, k_dim)

print("Cell 2 loaded: augment_batch() ready. "
      "NOTE: noise applied in PCA-component space (documented deviation from "
      "raw-band-space spec — see Phase 4 notes).")

Cell 2 loaded: augment_batch() ready. NOTE: noise applied in PCA-component space (documented deviation from raw-band-space spec — see Phase 4 notes).


In [3]:
# CELL 3 — Model, training loop (reused unchanged from Phase 2/3, both fixes locked)
import pennylane as qml

QUANTUM_DEVICE_NAME = "default.qubit"
DIFF_METHOD = "backprop"

def build_quantum_layer():
    dev = qml.device(QUANTUM_DEVICE_NAME, wires=N_QUBITS)
    weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=ENTANGLING_LAYERS, n_wires=N_QUBITS)
    @qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
        return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]
    return qml.qnn.TorchLayer(circuit, {"weights": weight_shape})

class QuantumTokenEncoder(nn.Module):
    def __init__(self, k_dim):
        super().__init__()
        self.angle_proj = nn.Linear(k_dim, N_QUBITS)
        self.q_layer = build_quantum_layer()
        self.out_proj = nn.Linear(N_QUBITS, D_MODEL)
    def forward(self, tokens):
        b, n, k = tokens.shape
        theta = math.pi * torch.tanh(self.angle_proj(tokens))
        flat = theta.reshape(b * n, N_QUBITS)
        q_out = self.q_layer(flat).reshape(b, n, N_QUBITS)
        return self.out_proj(q_out)

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_tokens, d_model):
        super().__init__()
        pe = torch.zeros(n_tokens, d_model)
        pos = torch.arange(0, n_tokens, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe

class QuantFormer(nn.Module):
    def __init__(self, k_dim, n_classes):
        super().__init__()
        self.q_encoder = QuantumTokenEncoder(k_dim)
        self.pos_enc = SinusoidalPositionalEncoding(N_TOKENS, D_MODEL)
        self.encoder = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True, dropout=0.0)
        self.classifier = nn.Linear(D_MODEL, n_classes)
    def forward(self, tokens):
        x = self.q_encoder(tokens)
        x = self.pos_enc(x)
        x = self.encoder(x)
        return self.classifier(x.mean(dim=1))

def get_param_groups(model):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        (no_decay if "q_layer" in name else decay).append(param)
    return [{"params": decay, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0}]

def train_one_seed(seed, k_dim, n_classes, train_tokens, train_labels, val_tokens, val_labels,
                    augment=False, aug_seed=None):
    torch.manual_seed(seed)
    model = QuantFormer(k_dim, n_classes).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    best_val_acc, best_state = -1, None
    rng = np.random.default_rng(aug_seed) if augment else None

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        xb_epoch = train_tokens
        if augment:
            xb_epoch = augment_batch(train_tokens, k_dim, rng)
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = xb_epoch[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_acc = accuracy_score(val_labels, model(val_tokens.to(DEVICE)).argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc, best_state = val_acc, {k: v.clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 10 == 0:
            print(f"    epoch={epoch+1}/{EPOCHS} val_acc={val_acc:.4f} (best={best_val_acc:.4f})")

    model.load_state_dict(best_state)
    return model

def evaluate(model, test_tokens, test_labels, n_classes):
    model.eval()
    with torch.no_grad():
        preds = model(test_tokens.to(DEVICE)).argmax(dim=1).cpu().numpy()
    labels_np = test_labels.numpy()
    cm = confusion_matrix(labels_np, preds, labels=list(range(n_classes)))
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    return {"OA": accuracy_score(labels_np, preds), "AA": per_class_acc.mean(),
            "kappa": cohen_kappa_score(labels_np, preds)}

print("Cell 3 loaded: model + training (dropout=0, best-checkpoint) ready.")

Cell 3 loaded: model + training (dropout=0, best-checkpoint) ready.


In [4]:
# CELL 4 — Phase 4 main run: Salinas, KSC, Botswana, Full config only
all_results = {}

for name in EXTENSION_DATASETS:
    ds = datasets[name]
    all_results[name] = {}
    for augment in [False, True]:
        tag = "augmented" if augment else "unaugmented"
        all_results[name][tag] = []
        for seed in SEEDS:
            print(f"\n=== {name} [{tag}], seed={seed} ===")
            start = time.time()
            model = train_one_seed(
                seed, ds["k_dim"], ds["n_classes"],
                ds["train_tokens"], ds["train_labels"], ds["val_tokens"], ds["val_labels"],
                augment=augment, aug_seed=seed + 1000,
            )
            elapsed = time.time() - start
            metrics = evaluate(model, ds["test_tokens"], ds["test_labels"], ds["n_classes"])
            metrics["seed"] = seed
            metrics["train_time_sec"] = elapsed
            all_results[name][tag].append(metrics)
            print(f"  OA={metrics['OA']:.4f} AA={metrics['AA']:.4f} kappa={metrics['kappa']:.4f} "
                  f"({elapsed/60:.1f} min)")

print(f"\n{'='*60}\nPHASE 4 SUMMARY\n{'='*60}")
for name in EXTENSION_DATASETS:
    for tag in ["unaugmented", "augmented"]:
        oas = [r["OA"] for r in all_results[name][tag]]
        print(f"{name} [{tag}]: OA = {np.mean(oas):.4f} ± {np.std(oas):.4f}")

with open("phase4_extension_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved phase4_extension_results.json")


=== Salinas [unaugmented], seed=42 ===
    epoch=10/50 val_acc=0.9294 (best=0.9760)
    epoch=20/50 val_acc=0.9621 (best=0.9887)
    epoch=30/50 val_acc=0.9891 (best=0.9891)
    epoch=40/50 val_acc=0.9841 (best=0.9891)
    epoch=50/50 val_acc=0.9874 (best=0.9893)
  OA=0.9895 AA=0.9918 kappa=0.9883 (9.1 min)

=== Salinas [unaugmented], seed=43 ===
    epoch=10/50 val_acc=0.9658 (best=0.9658)
    epoch=20/50 val_acc=0.9625 (best=0.9900)
    epoch=30/50 val_acc=0.9913 (best=0.9919)
    epoch=40/50 val_acc=0.9693 (best=0.9919)
    epoch=50/50 val_acc=0.9939 (best=0.9939)
  OA=0.9936 AA=0.9961 kappa=0.9929 (9.5 min)

=== Salinas [unaugmented], seed=44 ===
    epoch=10/50 val_acc=0.9549 (best=0.9717)
    epoch=20/50 val_acc=0.9887 (best=0.9887)
    epoch=30/50 val_acc=0.9836 (best=0.9887)
    epoch=40/50 val_acc=0.9817 (best=0.9933)
    epoch=50/50 val_acc=0.9631 (best=0.9948)
  OA=0.9947 AA=0.9965 kappa=0.9941 (9.5 min)

=== Salinas [augmented], seed=42 ===
    epoch=10/50 val_acc=0.9174 (

In [ ]:
# Cell 5 - Pull the complete Phase 4 summary directly from the saved file
with open("phase4_extension_results.json") as f:
    saved = json.load(f)

print(f"{'Dataset':<12} {'Condition':<14} {'OA mean':>10} {'OA std':>10}")
print("-" * 50)
for name in EXTENSION_DATASETS:
    for tag in ["unaugmented", "augmented"]:
        oas = [r["OA"] for r in saved[name][tag]]
        print(f"{name:<12} {tag:<14} {np.mean(oas):>10.4f} {np.std(oas):>10.4f}")

Dataset      Condition         OA mean     OA std
--------------------------------------------------
Salinas      unaugmented        0.9926     0.0022
Salinas      augmented          0.9890     0.0005
KSC          unaugmented        0.9647     0.0099
KSC          augmented          0.9756     0.0053
Botswana     unaugmented        0.9449     0.0100
Botswana     augmented          0.9608     0.0160
